### 1. Install Pytorch-FID

In [1]:
%pip install pytorch-fid

Note: you may need to restart the kernel to use updated packages.


### 2. Data Preparation 

#### 2-1 Python Package

In [ ]:
# pytorch
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
# torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
# pillow(PIL)
from PIL import Image
# python standard library
import os
import math
import glob
from pathlib import Path
from types import SimpleNamespace

#### 2-2 Construct the functions (Preparations/ UNet/ Diffusion process/ Training/ Sampling)

In [ ]:
# ============================================ 1. Preparations ============================================
def exists(x): return x is not None

def sinusoidal_time_embedding(t, dim):
    """
    Sinusoidal position embedding for timestep t.
    """
    device, half = t.device, dim // 2
    freqs = torch.exp(
        torch.linspace(math.log(1e-4), math.log(1.0), half, device=device)
    )
    # Outer product
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    
    # sin + cos
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
    return emb 

# ============================================ 2. Small UNet for 28x28 ===================================
class ResBlock(nn.Module):
    """
    Residual block with timestep conditioning.
    """
    def __init__(self, c_in, c_out, t_dim):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_dim, c_out)
        )
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        # If channel size mismatches, use 1x1 conv as shortcut
        self.skip  = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = F.silu(h + self.time_mlp(t_emb)[:, :, None, None])
        h = self.conv2(h)
        return F.silu(h + self.skip(x))

class UNetSmall(nn.Module):
    """
    A simplified UNet for MNIST (28x28).
    No attention, only down -> bottleneck -> up.
    """
    def __init__(self, in_ch=3, base=192, t_dim=128):
        super().__init__()
        self.t_mlp = nn.Sequential(
            nn.Linear(128, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim)
        )
        # ---------------- Downsampling ----------------
        self.rb1 = ResBlock(in_ch, base, t_dim)
        self.down1 = nn.Conv2d(base, base, 4, 2, 1)     # 28->14
        self.rb2 = ResBlock(base, base*2, t_dim)
        self.down2 = nn.Conv2d(base*2, base*2, 4, 2, 1) # 14->7
        
        # bottleneck
        self.rb3 = ResBlock(base*2, base*2, t_dim)

        # ---------------- Upsampling ------------------
        self.up1 = nn.ConvTranspose2d(base*2, base*2, 4, 2, 1)  # 7->14
        self.rb4 = ResBlock(base*2, base, t_dim)
        self.up2 = nn.ConvTranspose2d(base, base, 4, 2, 1)      # 14->28
        self.out = nn.Sequential(
            nn.Conv2d(base, in_ch, 3, padding=1)
        )

    def forward(self, x, t):  # predict epsilon
        t_emb = self.t_mlp(sinusoidal_time_embedding(t, 128))
        # -------Down path-------
        h1 = self.rb1(x, t_emb)
        h  = self.down1(h1)
        h2 = self.rb2(h, t_emb)
        h  = self.down2(h2)
        h  = self.rb3(h, t_emb)

        # -------Up path-------
        h  = self.up1(h)
        h  = self.rb4(h, t_emb)
        h  = self.up2(h)
        return self.out(h)

# ============================================ 3. Diffusion Process (Forward q & Reverse p) ==============
def make_beta_schedule(T=1000, start=1e-4, end=0.02):
    """
    Linear beta schedule: beta_1 ~ beta_T.
    """
    return torch.linspace(start, end, T)

class Diffusion:
    
    def __init__(self, T=1000, device="cuda"):
        self.device = device
        self.T = T
        betas = make_beta_schedule(T).to(device)
        alphas = 1. - betas
        self.alphas_bar = torch.cumprod(alphas, dim=0) 
        self.sqrt_ab = torch.sqrt(self.alphas_bar)
        self.sqrt_one_minus_ab = torch.sqrt(1. - self.alphas_bar)
        self.betas = betas
        self.alphas = alphas

    def q_sample(self, x0, t, eps=None):
        if not exists(eps):
            eps = torch.randn_like(x0)
        sqrt_ab_t = self.sqrt_ab[t].view(-1,1,1,1)
        sqrt_om_t = self.sqrt_one_minus_ab[t].view(-1,1,1,1)
        return sqrt_ab_t * x0 + sqrt_om_t * eps, eps

    @torch.no_grad()
    def p_sample_loop(self, model, n, cfg):
        x = torch.randn(n, 3, 28, 28, device=self.device)
        for t in reversed(range(self.T)):
            t_ = torch.full((n,), t, device=self.device, dtype=torch.long)
            eps_theta = model(x, t_)
            beta_t = self.betas[t]
            alpha_t = self.alphas[t]
            ab_t = self.alphas_bar[t]

            mu = (1. / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - ab_t)) * eps_theta)
            if t > 0:
                z = torch.randn_like(x)
                x = mu + torch.sqrt(beta_t) * z
            else:
                x = mu
        x = (x.clamp(-1, 1) + 1) / 2.0  # [-1,1] -> [0,1]
        return x

# ============================================ 4. Data Utilities =========================================
class PNGFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        patts = [os.path.join(root, "train", "*.png"),
                 os.path.join(root, "*.png")]
        files = []
        for p in patts:
            files.extend(glob.glob(p))
        self.files = sorted(files)
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        # label is not needed, return dummy 0
        return img, 0

def get_data_loader(root, batch_size):
    """
    MNIST loader with 28x28 -> 3 channels -> [-1,1] normalization.
    """
    tfm = transforms.Compose([
        transforms.Resize((28,28)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])
    ds = datasets.MNIST(root=root, train=True, download=True, transform=tfm)

    use_gpu = torch.cuda.is_available()
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4 if use_gpu else 0, 
        pin_memory=use_gpu,
        persistent_workers=use_gpu,        
        prefetch_factor=2 if use_gpu else None
    )

# ============================================ 5. Training ==============================================  
def train(args):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dl = get_data_loader(args.data, args.batch)
    model = UNetSmall().to(device)
    diff = Diffusion(T=args.steps, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)

    model.train()
    global_step = 0
    best_loss = float("inf")

    for epoch in range(args.epochs):
        print(f"Training epoch {epoch+1}/{args.epochs}")
        epoch_loss = 0
        
        for x0, _ in dl:
            x0 = x0.to(device)
            b = x0.size(0)
            t = torch.randint(0, diff.T, (b,), device=device, dtype=torch.long)
            x_t, eps = diff.q_sample(x0, t)
            eps_hat = model(x_t, t)
            loss = F.mse_loss(eps_hat, eps)
            epoch_loss += loss.item()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # stabilize training
            opt.step()
            global_step += 1
            
        #avg loss
        epoch_loss /= len(dl)
            
        print(f"Epoch {epoch+1}/{args.epochs} | loss={epoch_loss:.4f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            os.makedirs(args.ckpt_dir, exist_ok=True)
            torch.save(model.state_dict(), os.path.join(args.ckpt_dir, "best.pth"))
    torch.save(model.state_dict(), os.path.join(args.ckpt_dir, "last.pth"))
    print("Training Finished.")

# ============================================== 6. Sampling ============================================ 
def sample(args):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = UNetSmall().to(device)
    model.load_state_dict(torch.load(os.path.join(args.ckpt_dir, "best.pth"), map_location=device))
    model.eval()
    diff = Diffusion(T=args.steps, device=device)

    out_dir = Path(args.out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    total = args.num
    bs = args.s_batch
    saved = 0; idx = 1
    while saved < total:
        cur = min(bs, total - saved)
        imgs = diff.p_sample_loop(model, cur, None)  # [cur,3,28,28] in [0,1]
        imgs = imgs.clamp(0.0, 1.0)
        imgs = (imgs * 255).byte().cpu()
        for i in range(cur):
            fn = out_dir / f"{idx:05d}.png"
            Image.fromarray(imgs[i].permute(1,2,0).numpy()).save(fn)
            idx += 1
            if (idx % 1000) == 0:
                print(f"Sample {idx}/{total}")
        saved += cur
    print(f"Saved {saved} images to {str(out_dir)}")

### 3. Training

In [ ]:
args = SimpleNamespace(
    mode="train", data=".HW3_NTHU_114064545/mnist", ckpt_dir="./ckpt", out_dir="./img_114064545",
    epochs=150, batch=128, steps=1000, lr=8e-5, num=10000, s_batch=512
)
train(args)

Training epoch 1/150
Epoch 1/150 | loss=0.3944
Training epoch 2/150
Epoch 2/150 | loss=0.1046
Training epoch 3/150
Epoch 3/150 | loss=0.0562
Training epoch 4/150
Epoch 4/150 | loss=0.0437
Training epoch 5/150
Epoch 5/150 | loss=0.0344
Training epoch 6/150
Epoch 6/150 | loss=0.0302
Training epoch 7/150
Epoch 7/150 | loss=0.0261
Training epoch 8/150
Epoch 8/150 | loss=0.0239
Training epoch 9/150
Epoch 9/150 | loss=0.0223
Training epoch 10/150
Epoch 10/150 | loss=0.0213
Training epoch 11/150
Epoch 11/150 | loss=0.0208
Training epoch 12/150
Epoch 12/150 | loss=0.0203
Training epoch 13/150
Epoch 13/150 | loss=0.0196
Training epoch 14/150
Epoch 14/150 | loss=0.0189
Training epoch 15/150
Epoch 15/150 | loss=0.0190
Training epoch 16/150
Epoch 16/150 | loss=0.0184
Training epoch 17/150
Epoch 17/150 | loss=0.0180
Training epoch 18/150
Epoch 18/150 | loss=0.0180
Training epoch 19/150
Epoch 19/150 | loss=0.0176
Training epoch 20/150
Epoch 20/150 | loss=0.0173
Training epoch 21/150
Epoch 21/150 | l

### 4. Testing

In [14]:
args.mode = "sample"
sample(args)

Sample 1000/10000
Sample 2000/10000
Sample 3000/10000
Sample 4000/10000
Sample 5000/10000
Sample 6000/10000
Sample 7000/10000
Sample 8000/10000
Sample 9000/10000
Sample 10000/10000
Saved 10000 images to gen


### 5. FID Evaluation

- With the training dataset

In [ ]:
!python -m pytorch_fid /home/shangche45/HW3_NTHU_114064545/img_114064545 /home/shangche45/HW3_NTHU_114064545/mnist/train

100%|███████████████████████████████████████| 1200/1200 [01:17<00:00, 15.54it/s]
FID:  25.539056215003853


- With precalculated mean and covariance

In [ ]:
!python -m pytorch_fid /home/shangche45/HW3_NTHU_114064545/img_114064545 /home/shangche45/HW3_NTHU_114064545/mnist.npz

100%|█████████████████████████████████████████| 200/200 [00:13<00:00, 14.87it/s]
FID:  25.533290934402714


### 6. Diffusion process visualization

#### 6-1 Define the function

In [ ]:
@torch.no_grad()
def save_diffusion_process_grid(diff, model, num_samples=8, num_steps=8,
                                filename="diffusion_process.png"):
    """
    horizontal: num_samples of generated samples
    vertical: num_steps of timestep (noisy to clean)
    """
    device = diff.device
    model.eval()
    
    # Gaussian Noise
    x = torch.randn(num_samples, 3, 28, 28, device=device)
    
    # split the timestep to 7 equal parts (8 num_steps)
    # 8 timesteps: [999, 856, 713, 570, 427, 284, 141, 0]
    ts_to_save = torch.linspace(diff.T - 1, 0, num_steps,
                                dtype=torch.long, device=device)
    ts_set = set(ts_to_save.tolist())

    snapshots = []

    for t in reversed(range(diff.T)):
        t_tensor = torch.full((num_samples,), t, device=device, dtype=torch.long)
        eps_theta = model(x, t_tensor)

        beta_t = diff.betas[t]
        alpha_t = diff.alphas[t]
        ab_t = diff.alphas_bar[t]

        mu = (1. / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - ab_t)) * eps_theta)
        if t > 0:
            x = mu + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mu

        if float(t) in ts_set:
            x_vis = (x.clamp(-1, 1) + 1) / 2.0   # [-1,1] → [0,1]
            snapshots.append(x_vis.detach().cpu()) 

    imgs = torch.cat(snapshots, dim=0)

    #Arrange these 64 images into a form
    grid = make_grid(imgs, nrow=num_samples, padding=2)

    save_image(grid, filename)
    print(f"Saved diffusion process grid to {filename}")

#### 6-2 Save diffusion process grid

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

diff = Diffusion(T=1000, device=device)

model = UNetSmall().to(device)
model.load_state_dict(torch.load("/home/shangche45/HW3_NTHU_114064545/ckpt/best.pth", map_location=device))

save_diffusion_process_grid(diff, model,
                            num_samples=8,
                            num_steps=8,
                            filename="diffusion_process.png")

Saved diffusion process grid to diffusion_process.png
